# Global fraud-CDF cube, by alarm split

Estimate `F(t) = P(x <= t | is_fraud = 1)` on **all train fraud**. Apply that map to every row. Then look at the cube `(F(age), F(merchant_risk), F(velocity))` inside no-alarm and inside alarm, colored by label.

The scatter subsamples non-fraud so the cloud is readable. Occupancy of the origin and far corner uses every row.

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
from ccf_ml import file_utils
from ccf_ml.cross_validation import alarm as alarm_mod
from ccf_ml.features import fraud_ecdf as fraud_ecdf_mod
importlib.reload(file_utils)
importlib.reload(alarm_mod)
importlib.reload(fraud_ecdf_mod)

from ccf_ml.cross_validation.alarm import alarm_flag_mask
from ccf_ml.cross_validation.constants import (
    ALARM_FLAG_COLUMNS,
    TARGET_COLUMN,
)
from ccf_ml.features.fraud_ecdf import (
    FraudEcdf,
    fraud_ecdf_from_frame,
    transform_frame,
)
from ccf_ml.file_utils import (
    get_script_plot_dir,
    get_script_table_dir,
    get_train_csv_path,
)
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DELIMITER_VALUE = "-" * 50
NOTEBOOK_STEM = "fraud_ecdf_3d_by_alarm"
FRAUD_ECDF_COLUMNS = (
    "customer_age",
    "merchant_risk_score",
    "velocity_score",
)
ALARM_SPLIT_COL = "alarm_flag_split"
ALARM_SPLIT_LABELS = {
    False: "no alarm",
    True: "at least one alarm",
}
ALARM_SPLIT_ORDER = ["no alarm", "at least one alarm"]
TARGET_GIVEN_LABELS = {0: "not fraud", 1: "fraud"}
MAX_NON_FRAUD_SCATTER = 30000
SCATTER_SEED = 42
LOW_CUBE = 0.1
HIGH_CUBE = 0.9
VIEWS = (
    ("origin", 22.0, -60.0),
    ("far_corner", 22.0, 120.0),
)
tables_dir = get_script_table_dir(NOTEBOOK_STEM)

frame = pd.read_csv(get_train_csv_path())
has_alarm = alarm_flag_mask(frame, columns=ALARM_FLAG_COLUMNS)
frame[ALARM_SPLIT_COL] = has_alarm.map(ALARM_SPLIT_LABELS)
n_columns = len(FRAUD_ECDF_COLUMNS)
curve_slots: list[FraudEcdf | None] = [None] * n_columns
for i in range(n_columns):
    name = FRAUD_ECDF_COLUMNS[i]
    curve_slots[i] = fraud_ecdf_from_frame(frame, column=name)
    curve = curve_slots[i]
    if curve is None:
        raise ValueError(f"missing curve for {name}")
    print(name, "n_fraud", curve.n)
curves = tuple(curve for curve in curve_slots if curve is not None)
if len(curves) != n_columns:
    raise ValueError("failed to estimate every fraud CDF")
cube = transform_frame(frame, curves=curves)
print("n_rows", len(cube))
print(cube[list(FRAUD_ECDF_COLUMNS) + [TARGET_COLUMN, ALARM_SPLIT_COL]].head())


## Occupancy of the origin and the far corner

Origin: all three `F <= 0.1`. Far corner: all three `F >= 0.9`. Because `F` is the fraud ECDF, fraud would be about `0.1^3 = 0.001` in each cube if the three coordinates were independent. A pile of non-fraud in the origin means those rows sit on the low side of the fraud distribution on every axis at once.

In [ ]:
def cube_occupancy(
    part: pd.DataFrame,
    *,
    region: str,
    mask: np.ndarray,
) -> dict[str, object]:
    n = int(np.sum(mask))
    n_fraud = int(np.sum(mask & (part[TARGET_COLUMN].to_numpy() == 1)))
    n_part = len(part)
    n_part_fraud = int(part[TARGET_COLUMN].sum())
    rate = float(n_fraud / n) if n else float("nan")
    return {
        "region": region,
        "n": n,
        "n_fraud": n_fraud,
        "rate": rate,
        "share_of_rows": n / n_part if n_part else float("nan"),
        "share_of_fraud": (
            n_fraud / n_part_fraud if n_part_fraud else float("nan")
        ),
    }


n_splits = len(ALARM_SPLIT_ORDER)
n_regions = 3
occ_rows: list[dict[str, object] | None] = [None] * (n_splits * n_regions)
row_i = 0
for split_i in range(n_splits):
    name = ALARM_SPLIT_ORDER[split_i]
    part = cube.loc[cube[ALARM_SPLIT_COL] == name]
    vals = part.loc[:, list(FRAUD_ECDF_COLUMNS)].to_numpy(dtype=np.float64)
    all_low = np.all(vals <= LOW_CUBE, axis=1)
    all_high = np.all(vals >= HIGH_CUBE, axis=1)
    whole = np.ones(len(part), dtype=bool)
    regions = (
        ("all rows", whole),
        (f"all F <= {LOW_CUBE}", all_low),
        (f"all F >= {HIGH_CUBE}", all_high),
    )
    for region_i in range(n_regions):
        region_name, mask = regions[region_i]
        metrics = cube_occupancy(part, region=region_name, mask=mask)
        metrics[ALARM_SPLIT_COL] = name
        occ_rows[row_i] = metrics
        row_i += 1
occupancy = pd.DataFrame([row for row in occ_rows if row is not None])
occupancy = occupancy.set_index([ALARM_SPLIT_COL, "region"])
print(occupancy.round(4))
occ_path = tables_dir / "extreme_cube_occupancy.csv"
occupancy.to_csv(occ_path)
print("saved", occ_path)


## 3D scatter

Non-fraud is drawn first (subsampled to 30k). Fraud is drawn on top (every fraud row in the split). Two camera angles: toward the origin, then toward `(1, 1, 1)`.

In [ ]:
SCATTER_SECTION = "scatter3d"
AGE_COL, RISK_COL, VEL_COL = FRAUD_ECDF_COLUMNS
rng = np.random.default_rng(SCATTER_SEED)

def scatter_sample(part: pd.DataFrame) -> pd.DataFrame:
    is_fraud = part[TARGET_COLUMN].to_numpy() == 1
    fraud = part.loc[is_fraud]
    non_fraud = part.loc[~is_fraud]
    n_non = len(non_fraud)
    if n_non > MAX_NON_FRAUD_SCATTER:
        chosen = rng.choice(n_non, size=MAX_NON_FRAUD_SCATTER, replace=False)
        non_fraud = non_fraud.iloc[chosen]
    return pd.concat([non_fraud, fraud], ignore_index=True)


def plot_split_scatter(
    part: pd.DataFrame,
    *,
    title: str,
    elev: float,
    azim: float,
) -> tuple:
    sample = scatter_sample(part)
    fig = plt.figure(figsize=(8.5, 7.0))
    ax = fig.add_subplot(111, projection="3d")
    levels = (0, 1)
    colors = {0: "#4c78a8", 1: "#e45756"}
    sizes = {0: 4.0, 1: 8.0}
    alphas = {0: 0.12, 1: 0.45}
    n_levels = len(levels)
    for i in range(n_levels):
        level = levels[i]
        pts = sample.loc[sample[TARGET_COLUMN] == level]
        ax.scatter(
            pts[AGE_COL].to_numpy(),
            pts[RISK_COL].to_numpy(),
            pts[VEL_COL].to_numpy(),
            c=colors[level],
            s=sizes[level],
            alpha=alphas[level],
            linewidths=0.0,
            label=TARGET_GIVEN_LABELS[level],
            depthshade=False,
        )
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_zlim(0.0, 1.0)
    ax.set_xlabel(f"F({AGE_COL})")
    ax.set_ylabel(f"F({RISK_COL})")
    ax.set_zlabel(f"F({VEL_COL})")
    ax.view_init(elev=elev, azim=azim)
    ax.legend(loc="upper left")
    ax.set_title(title)
    fig.tight_layout()
    return fig, ax


n_views = len(VIEWS)
for split_i in range(n_splits):
    name = ALARM_SPLIT_ORDER[split_i]
    part = cube.loc[cube[ALARM_SPLIT_COL] == name]
    stem = name.replace(" ", "_")
    print(DELIMITER_VALUE)
    print(name, "n", len(part), "n_fraud", int(part[TARGET_COLUMN].sum()))
    for view_i in range(n_views):
        view_name, elev, azim = VIEWS[view_i]
        fig, ax = plot_split_scatter(
            part,
            title=f"{name} ({view_name})",
            elev=elev,
            azim=azim,
        )
        plot_path = (
            get_script_plot_dir(NOTEBOOK_STEM, section=SCATTER_SECTION)
            / f"{stem}_{view_name}.png"
        )
        fig.savefig(plot_path, dpi=140)
        print("saved", plot_path)
        plt.show()
